In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules

In [2]:
from ortools.linear_solver import pywraplp
import time

In [3]:
import json


if IN_COLAB:
  import gdown
  !pip install -q ortools
  FILE_ID = '1Ytjbyrt_dfH17cqR8goRV5WSYO4oblHE'

  url = f'https://drive.google.com/uc?id={FILE_ID}'
  output = '01.json'

  gdown.download(url, output, quiet=False)

else:
  output = "../data/instances/01.json"

with open(output, 'r', encoding='utf-8') as file:
  data = json.load(file)
  print(json.dumps(data, indent=2)[:500] + "... [Truncated]")

{
  "number_of_days": 28,
  "skills": [
    "Nurse",
    "HeadNurse"
  ],
  "shifts": [
    {
      "shift_id": "E",
      "needed_skill": "Nurse",
      "demand": [
        8,
        8,
        8,
        8,
        8,
        6,
        6,
        8,
        8,
        8,
        8,
        8,
        6,
        6,
        8,
        8,
        8,
        8,
        8,
        6,
        6,
        8,
        8,
        8,
        8,
        8,
        6,
        6
      ]
    },
    {
      ... [Truncated]


In [9]:
D = range(data["number_of_days"])
nurses = data["nurses"]
shifts = data["shifts"]    

N = [nu["nurse_id"] for nu in nurses]
S = [sh["shift_id"] for sh in shifts]    
skills_n = {nu["nurse_id"]: set(nu["skills"]) for nu in nurses}
needed = {sh["shift_id"]: sh["needed_skill"] for sh in shifts}
print(needed)

{'E': 'Nurse', 'L': 'Nurse', 'D': 'Nurse', 'N': 'Nurse', 'DH': 'HeadNurse'}


In [10]:
solver = pywraplp.Solver.CreateSolver('GLOP')

In [ ]:
#### VARIABLES


### 1) BINARY VARIABLES
x = {}
for n in N:
    for d in D:
        for s in S:
            if needed[s] in skills_n[n]:
                x[n, d, s] = solver.BoolVar(f"1 if nurse {n} is assigned to shift {s} on day {d}, 0 otherwise")

w={}
for n in N:
    for d in D:
        w[n,d] = solver.BoolVar(f"1 if nurse {n} works on day {d}, 0 otherwise")

### 2) CONTINUOUS VARAIBLES

o = {}
u = {}
for s in S:
    for d in D:
        o[s, d] = solver.NumVar(0, solver.infinity(), f"Overstaffing of shift {s} on day {d}")
        u[s, d] = solver.NumVar(0, solver.infinity(), f"Understaffing of shift {s} on day {d}") 